In [2]:
import pandas as pd
import re
from dateutil import parser

# load your file
df = pd.read_csv('../amazon_india_2016.csv', low_memory=False)

# cleaning functions
def clean_price(val):
    val = str(val).replace('₹', '').replace(',', '').strip()
    try: return float(val)
    except: return None

def clean_rating(val):
    if pd.isna(val): return None
    val = str(val).lower().strip()
    if 'star' in val: return float(re.search(r'\d+\.?\d*', val).group())
    if '/' in val: a, b = val.split('/'); return round(float(a)/float(b)*5, 2)
    try: return float(val)
    except: return None

def clean_date(val):
    try: return parser.parse(val, dayfirst=True).strftime('%Y-%m-%d')
    except: return None

def clean_delivery(val):
    val = str(val).strip().lower()
    if val == 'same day': return 0
    if val == 'express': return 1
    if '-' in val:
        try:
            a, b = val.split('-')
            return round((float(a) + float(b.split()[0])) / 2)
        except: return None
    try:
        val = int(float(val))
        if val < 0: return None
        if val > 30: return None
        return val
    except: return None

# apply cleaning
true_values = ['True', 'TRUE', '1', 'Yes']
manual_mapping = {'Bombay': 'Mumbai', 'Madras': 'Chennai', 'Bengaluru': 'Bangalore', 'Calcutta': 'Kolkata'}

df['original_price_inr'] = df['original_price_inr'].apply(clean_price)
df['product_rating'] = df['product_rating'].apply(clean_rating)
df['product_rating'] = df.groupby('category')['product_rating'].transform(lambda x: x.fillna(x.median()))
df['order_date'] = df['order_date'].apply(clean_date)
df['order_date'] = pd.to_datetime(df['order_date'])
df['delivery_days'] = df['delivery_days'].apply(clean_delivery)
df['delivery_days'] = df['delivery_days'].fillna(df['delivery_days'].median())
df['customer_age_group'] = df['customer_age_group'].fillna(df['customer_age_group'].mode()[0])
df['delivery_charges'] = df['delivery_charges'].fillna(0)
df['festival_name'] = df['festival_name'].fillna('No Festival')
df['is_prime_member'] = False
df['is_prime_eligible'] = df['is_prime_eligible'].isin(true_values)
df['customer_city'] = df['customer_city'].str.strip().str.title().replace(manual_mapping)
df['category'] = df['category'].replace({'ELECTRONICS': 'Electronics', 'Electronics & Accessories': 'Electronics', 'Electronic': 'Electronics', 'Electronicss': 'Electronics'})

# fix outlier prices
df['original_price_inr'] = df.apply(lambda row: row['discounted_price_inr'] / (1 - row['discount_percent']/100) if row['original_price_inr'] > row['discounted_price_inr'] * 10 else row['original_price_inr'], axis=1)
df.loc[df['original_price_inr'] < df['discounted_price_inr'], 'original_price_inr'] = df.apply(lambda row: row['discounted_price_inr'] / (1 - row['discount_percent']/100) if row['original_price_inr'] < row['discounted_price_inr'] else row['original_price_inr'], axis=1)

# remove duplicates
df = df.drop_duplicates(subset=['customer_id', 'product_id', 'order_date', 'final_amount_inr'], keep='first')

print(f"✅ Done! Shape: {df.shape}")
print(df.isnull().sum())

✅ Done! Shape: (54999, 34)
transaction_id                0
order_date                    0
customer_id                   0
product_id                    0
product_name                  0
category                      0
subcategory                   0
brand                         0
original_price_inr         1622
discount_percent              0
discounted_price_inr          0
quantity                      0
subtotal_inr                  0
delivery_charges              0
final_amount_inr              0
customer_city                 0
customer_state                0
customer_tier                 0
customer_spending_tier        0
customer_age_group            0
payment_method                0
delivery_days                 0
delivery_type                 0
is_prime_member               0
is_festival_sale              0
festival_name                 0
customer_rating           16678
return_status                 0
order_month                   0
order_year                    0
order_quarter

In [3]:
df['original_price_inr'] = df.apply(
    lambda row: row['discounted_price_inr'] / (1 - row['discount_percent']/100)
    if pd.isna(row['original_price_inr'])
    else row['original_price_inr'], axis=1
)

print(df['original_price_inr'].isna().sum())  # should be 0

0


In [4]:
df.isnull().sum()

transaction_id                0
order_date                    0
customer_id                   0
product_id                    0
product_name                  0
category                      0
subcategory                   0
brand                         0
original_price_inr            0
discount_percent              0
discounted_price_inr          0
quantity                      0
subtotal_inr                  0
delivery_charges              0
final_amount_inr              0
customer_city                 0
customer_state                0
customer_tier                 0
customer_spending_tier        0
customer_age_group            0
payment_method                0
delivery_days                 0
delivery_type                 0
is_prime_member               0
is_festival_sale              0
festival_name                 0
customer_rating           16678
return_status                 0
order_month                   0
order_year                    0
order_quarter                 0
product_

In [5]:
df.dtypes

transaction_id                    object
order_date                datetime64[ns]
customer_id                       object
product_id                        object
product_name                      object
category                          object
subcategory                       object
brand                             object
original_price_inr               float64
discount_percent                 float64
discounted_price_inr             float64
quantity                           int64
subtotal_inr                     float64
delivery_charges                 float64
final_amount_inr                 float64
customer_city                     object
customer_state                    object
customer_tier                     object
customer_spending_tier            object
customer_age_group                object
payment_method                    object
delivery_days                    float64
delivery_type                     object
is_prime_member                     bool
is_festival_sale

In [6]:
df.head(10)

,transaction_id,order_date,customer_id,product_id,product_name,category,subcategory,brand,original_price_inr,discount_percent,...,is_festival_sale,festival_name,customer_rating,return_status,order_month,order_year,order_quarter,product_weight_kg,is_prime_eligible,product_rating
0,TXN_2016_00000001,2016-01-27,CUST_2016_00002571,PROD_000124,Apple iPhone SE 16GB White,Electronics,Smartphones,Apple,101945.03,0.00,...,False,No Festival,NaN,Delivered,1,2016,1,0.23,True,4.6
1,TXN_2016_00000002,2016-07-01,CUST_2016_00003513,PROD_001612,Apple Pavilion 8GB RAM Silver,Electronics,Laptops,Apple,52750.75,14.72,...,False,No Festival,NaN,Delivered,1,2016,1,2.69,True,3.7
2,TXN_2016_00000003,2016-01-28,CUST_2015_00001993,PROD_001751,Realme Slate 4GB RAM Black,Electronics,Tablets,Realme,18238.60,0.00,...,False,No Festival,5.0,Delivered,1,2016,1,0.59,True,3.9
3,TXN_2016_00000004,2016-01-25,CUST_2016_00003593,PROD_000154,Samsung Galaxy J7 Prime 16GB Gold,Electronics,Smartphones,Samsung,33118.39,44.96,...,True,Republic Day Sale,4.5,Delivered,1,2016,1,0.17,True,3.4
4,TXN_2016_00000005,2016-01-26,CUST_2016_00015048,PROD_001710,Lenovo Tab M10 8GB RAM Black,Electronics,Tablets,Lenovo,59718.16,49.77,...,True,Republic Day Sale,NaN,Delivered,1,2016,1,0.49,True,4.1
5,TXN_2016_00000006,2016-05-01,CUST_2016_00012117,PROD_000034,Samsung Galaxy S6 Edge 16GB Blue,Electronics,Smartphones,Samsung,96021.61,28.37,...,False,No Festival,4.5,Returned,1,2016,1,0.18,True,3.3
6,TXN_2016_00000007,2016-02-01,CUST_2015_00007223,PROD_001935,Garmin Watch Deluxe,Electronics,Smart Watch,Garmin,46161.87,27.50,...,False,No Festival,4.0,Delivered,1,2016,1,0.08,False,4.8
7,TXN_2016_00000008,2016-01-18,CUST_2015_00003084,PROD_000058,OnePlus OnePlus X 16GB Black,Electronics,Smartphones,OnePlus,101282.29,5.71,...,False,No Festival,NaN,Delivered,1,2016,1,0.16,True,3.7
8,TXN_2016_00000009,2016-01-13,CUST_2015_00004286,PROD_000090,Motorola Moto G (3rd Gen) 32GB Blue,Electronics,Smartphones,Motorola,24101.70,0.00,...,FALSE,No Festival,4.5,Delivered,1,2016,1,0.20,True,3.8
9,TXN_2016_00000010,2016-03-01,CUST_2016_00013276,PROD_000132,Samsung Galaxy S7 Edge 16GB Black,Electronics,Smartphones,Samsung,101166.56,0.00,...,False,No Festival,3.5,Delivered,1,2016,1,0.20,True,4.4


In [7]:
def clean_rating(val):
    if pd.isna(val): return None
    val = str(val).lower().strip()
    if 'star' in val: return float(re.search(r'\d+\.?\d*', val).group())
    if '/' in val: a, b = val.split('/'); return round(float(a)/float(b)*5, 2)
    try: return float(val)
    except: return None

df['customer_rating'] = df['customer_rating'].apply(clean_rating)
df['customer_rating'] = df['customer_rating'].fillna(df['customer_rating'].median())

print(df['customer_rating'].isna().sum())  # should be 0

0


In [8]:
df.order_date.unique()

<DatetimeArray>
['2016-01-27 00:00:00', '2016-07-01 00:00:00', '2016-01-28 00:00:00',
 '2016-01-25 00:00:00', '2016-01-26 00:00:00', '2016-05-01 00:00:00',
 '2016-02-01 00:00:00', '2016-01-18 00:00:00', '2016-01-13 00:00:00',
 '2016-03-01 00:00:00',
 ...
 '2016-12-30 00:00:00', '2016-12-13 00:00:00', '2016-12-25 00:00:00',
 '2016-12-29 00:00:00', '2016-12-18 00:00:00', '2016-12-26 00:00:00',
 '2016-12-28 00:00:00', '2016-12-17 00:00:00', '2016-12-16 00:00:00',
 '2016-12-15 00:00:00']
Length: 365, dtype: datetime64[ns]

In [11]:
from sqlalchemy import create_engine


In [12]:
df.to_csv('2016_amazon.csv', index=False)
print("✅ CSV saved!")

engine = create_engine('mysql+pymysql://root:@127.0.0.1:3306/amazon_db?unix_socket=/Applications/XAMPP/xamppfiles/var/mysql/mysql.sock')
df.to_sql('transactions_2016', engine, if_exists='replace', index=False)
print("✅ Saved to MySQL as transactions_2016!")

✅ CSV saved!
✅ Saved to MySQL as transactions_2016!


In [13]:
df.original_price_inr.unique()

array([101945.03      ,  52750.75      ,  18238.6       , ...,
        70051.73735625,  18346.21594349,  19716.10933814], shape=(1461,))

In [14]:
df.isnull().sum()

transaction_id            0
order_date                0
customer_id               0
product_id                0
product_name              0
category                  0
subcategory               0
brand                     0
original_price_inr        0
discount_percent          0
discounted_price_inr      0
quantity                  0
subtotal_inr              0
delivery_charges          0
final_amount_inr          0
customer_city             0
customer_state            0
customer_tier             0
customer_spending_tier    0
customer_age_group        0
payment_method            0
delivery_days             0
delivery_type             0
is_prime_member           0
is_festival_sale          0
festival_name             0
customer_rating           0
return_status             0
order_month               0
order_year                0
order_quarter             0
product_weight_kg         0
is_prime_eligible         0
product_rating            0
dtype: int64

In [15]:
df.is_festival_sale.unique()

array(['False', 'True', 'FALSE', 'No', 'Yes', 'TRUE', '1', '0'],
      dtype=object)

In [16]:
df.payment_method.unique()

array(['COD', 'Credit Card', 'Debit Card', 'Net Banking', 'UPI'],
      dtype=object)

In [17]:
df = pd.read_csv('2016_amazon.csv', low_memory=False)

true_values = ['True', 'TRUE', '1', 'Yes']
df['is_festival_sale'] = df['is_festival_sale'].isin(true_values)

payment_mapping = {
    'Cash on Delivery': 'COD', 'C.O.D': 'COD',
    'CREDIT CARD': 'Credit Card', 'CC': 'Credit Card',
    'DEBIT CARD': 'Debit Card',
    'NET BANKING': 'Net Banking',
    'UPI': 'UPI', 'PhonePe': 'UPI', 'GooglePay': 'UPI', 'GPay': 'UPI'
}
df['payment_method'] = df['payment_method'].replace(payment_mapping)

print(df['is_festival_sale'].value_counts())
print(df['payment_method'].value_counts())

df.to_csv('2016_amazon.csv', index=False)
engine = create_engine('mysql+pymysql://root:@127.0.0.1:3306/amazon_db?unix_socket=/Applications/XAMPP/xamppfiles/var/mysql/mysql.sock')
df.to_sql('transactions_2016', engine, if_exists='replace', index=False)
print("✅ 2016 re-saved with all fixes!")

is_festival_sale
False    37634
True     17365
Name: count, dtype: int64
payment_method
COD            38533
Credit Card     7754
Debit Card      5433
Net Banking     2145
UPI             1134
Name: count, dtype: int64
✅ 2016 re-saved with all fixes!


In [18]:
df.is_festival_sale.unique()

array([False,  True])